# PyMolFit 01: Quickstart

This notebook performs one complete telluric correction on a reduced HARPS spectrum around the Na D doublet. It loads a managed AER line window, builds a MIPAS plus seasonal-GDAS atmosphere from the FITS metadata, fits the atmospheric and instrumental parameters, saves a provenance-bearing product, and compares the spectrum before and after correction.

The fit intervals and astrophysical masks are chosen from the known Na D region, not from residuals produced by PyMolFit. The result is a worked example, not a universal configuration for every instrument.

## Installation and first run

Install the plotting extra with `python -m pip install "pymolfit[plot]"`. The first scientific run may download and verify the official AER 3.9 catalogue. No HITRAN account or API key is required for the managed AER path used here.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from pymolfit import correct_file, load_spectrum, plot_fit

candidates = (Path.cwd() / "tutorials", Path.cwd())
TUTORIAL_ROOT = next(
    (path.resolve() for path in candidates if (path / "data").is_dir()),
    None,
)
if TUTORIAL_ROOT is None:
    raise FileNotFoundError("Open this notebook from the PyMolFit repository or tutorials directory")

INPUT = TUTORIAL_ROOT / "data" / "harps_nad_crop_air.fits"
OUTPUT = TUTORIAL_ROOT / "outputs"
OUTPUT.mkdir(exist_ok=True)
print("Input:", INPUT)

## Inspect the input

PyMolFit accepts common FITS tables, ECSV/CSV tables, numeric text, and one-dimensional FITS images with linear wavelength WCS. This FITS table already uses microns and standard-air wavelengths. `wavelength_medium="air"` describes the input; it does not blindly convert an already-air product.

In [ ]:
spectrum = load_spectrum(INPUT, wavelength_medium="air")
spectrum_angstrom = spectrum.to_air().to_unit("angstrom")

print("Pixels:", spectrum_angstrom.wavelength.size)
print("Range [Angstrom]:", spectrum_angstrom.wavelength[[0, -1]])
print("Valid pixels:", np.count_nonzero(spectrum_angstrom.valid))

plt.figure(figsize=(11, 4))
plt.plot(spectrum_angstrom.wavelength, spectrum_angstrom.flux, color="black", linewidth=0.8)
plt.xlabel("Air wavelength [Angstrom]")
plt.ylabel("Flux")
plt.title("Input HARPS spectrum around Na D")
plt.tight_layout()
plt.show()

## Fit and correct

The broad Na D regions are excluded because the stellar and circumstellar Na absorption is not part of the atmospheric forward model. A one-pixel box represents detector-pixel integration. Gaussian and Lorentz components describe the instrumental line-spread function; their values below are starting points and are refined within explicit bounds. The seasonal GDAS mode is used here for a deterministic tutorial that does not depend on network access. Tutorial 04 demonstrates time-local GDAS.

In [ ]:
result = correct_file(
    input_path=INPUT,
    output_path=OUTPUT / "01_beta_pic_corrected.ecsv",
    product_path=OUTPUT / "01_beta_pic_fit_product.ecsv",
    wavelength_medium="air",
    aer_catalog="auto",
    atmosphere_mode="mipas_gdas",
    gdas_mode="average",
    continuum_order=2,
    solve_continuum_linear=True,
    lsf_box_width_pixels=1.0,
    lsf_sigma_pixels=2.17,
    fit_lsf_sigma=True,
    lsf_sigma_bounds=(0.5, 4.0),
    lsf_lorentz_fwhm_pixels=0.5,
    fit_lsf_lorentz_fwhm=True,
    lsf_lorentz_fwhm_bounds=(0.0, 6.0),
    fit_wavelength_shift=True,
    wavelength_shift_bounds=(-2e-5, 2e-5),
    fit_ranges=((0.58825, 0.59065),),
    exclude_ranges=((0.58888, 0.58912), (0.58948, 0.58978)),
)

## Examine the result

`success=True` only means that the numerical optimizer terminated successfully. Also inspect alignment, residual structure, parameter bounds, transmission depth, and preservation of astrophysical features.

In [ ]:
print("Fit success:", result.success)
print("Fit message:", result.message)
print("Species scales:", result.species_scales)
print("Wavelength shift [Angstrom]:", result.wavelength_shift * 1e4)
print("Gaussian sigma [pixels]:", result.lsf_sigma_pixels)
print("Lorentz FWHM [pixels]:", result.lsf_lorentz_fwhm_pixels)
print("Parameters at bounds:", result.parameter_bound_status or "none")

plot_fit(result)
plt.show()

In [ ]:
observed = result.spectrum.to_air().to_unit("angstrom")
corrected = result.corrected.to_air().to_unit("angstrom")
window = (observed.wavelength >= 5887.5) & (observed.wavelength <= 5898.8)
scale = np.nanmedian(observed.flux[window])

plt.figure(figsize=(11, 4))
plt.plot(observed.wavelength, observed.flux / scale, color="black", alpha=0.65, label="Observed")
plt.plot(corrected.wavelength, corrected.flux / scale, color="tab:blue", label="Telluric corrected")
plt.axvspan(5888.8, 5891.2, color="tab:red", alpha=0.10, label="Excluded from fit")
plt.axvspan(5894.8, 5897.8, color="tab:red", alpha=0.10)
plt.xlim(5887.5, 5898.8)
plt.xlabel("Air wavelength [Angstrom]")
plt.ylabel("Flux / local median")
plt.title("Before and after telluric correction")
plt.legend()
plt.tight_layout()
plt.show()

## What was saved?

`01_beta_pic_corrected.ecsv` is the corrected spectrum. `01_beta_pic_fit_product.ecsv` additionally retains the input flux, fitted model, continuum, transmission, masks, fit parameters, bound flags, and machine-readable provenance. Keep the full fit product when a result must be audited or reproduced.

Continue with Tutorial 02 to see why each major fit control was introduced.